# Demonstrating Vector Resilience Under Real-World Flight Dynamics
This presentation module tests the AI against **FD002**. Unlike standard benchmarks, FD002 forces the neural net through **six unique operational regimes** (shifting altitudes, external temperatures, and Mach speeds represented via OpSet1-3). Our objective is to prove the RMSE stability of the PINN-LSTM combination as conditions dynamically shift.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error
import tensorflow as tf

sys.path.append(os.path.abspath('..'))
from src.data_processor import VectorSequenceProcessor

In [ ]:
# 1. Setup Processor & Extract FD002 Data
processor = VectorSequenceProcessor()
test_data_path = '../data/test_FD002.txt'

if os.path.exists(test_data_path):
    print("[1] Loading multidimensional flight profile (FD002)...")
    df_test = processor.load_data(test_data_path)
    df_test = processor.calculate_rul(df_test) # Using integrated dynamic target proxy

    # Standard aircraft engines cycle through 6 distinct 'OpSet' combinations here.
    # We map this into explicit clusters using SKLearn to group these 6 modes.
    kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
    df_test['Flight_Regime'] = kmeans.fit_predict(df_test[['OpSet1', 'OpSet2', 'OpSet3']])
    
    # Scale data
    df_test = processor.normalize_sensors(df_test, fit=True)

    # Generate Sequences tracking explicit regimes
    print("[2] Constructing bounded sequences...")
    window_size = 50
    seq_list, lbl_list, regime_list = [], [], []
    
    for _, group in df_test.groupby('Engine_ID'):
        sens_v = group[processor.sensor_cols].values
        rul_v = group['RUL'].values
        regime_v = group['Flight_Regime'].values
        
        for i in range(len(group) - window_size + 1):
            seq_list.append(sens_v[i : i + window_size])
            lbl_list.append(rul_v[i + window_size - 1])
            regime_list.append(regime_v[i + window_size - 1])
            
    X_test = np.array(seq_list)
    y_test = np.array(lbl_list)
    c_test = np.array(regime_list)
    print(f"✅ Successfully packaged {X_test.shape[0]} sequences.")
else:
    print(f"⚠️ Error: Could not locate {test_data_path}")

In [ ]:
# 2 & 3. Boot LSTM Weights and Evaluate Accuracies across Regimes
model_path = '../models/vxp2_lstm_v1.h5'

if os.path.exists(model_path):
    print(f"[3] Linking weights from {model_path}...")
    model = tf.keras.models.load_model(model_path)
    
    print("[4] Broadcasting inference array...")
    predictions = model.predict(X_test, verbose=0).flatten()
    
    total_rmse = np.sqrt(mean_squared_error(y_test, predictions))
    print(f"\n► OVERALL CMAPSS FD002 RMSE: {total_rmse:.2f} Cycles")
else:
    print("⚠️ Model artifact missing. Run training notebook first.")

In [ ]:
# 4. Render Analytics and Regimental Variance Plot 
if 'predictions' in locals():
    regimes = np.unique(c_test)
    scores = []
    
    for r in regimes:
        idx = (c_test == r)
        score = np.sqrt(mean_squared_error(y_test[idx], predictions[idx]))
        scores.append(score)
        
    plt.figure(figsize=(12, 7))
    sns.set_style("whitegrid")
    
    ax = sns.barplot(
        x=regimes,
        y=scores, 
        hue=regimes, 
        palette='mako', 
        legend=False
    )
    
    plt.title('Vector Prediction Error Under Diverse Operating Settings', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('K-Means Operating Mode Group (OpSet 1 to 3)', fontsize=14, labelpad=10)
    plt.ylabel('Root Mean Square Error (RMSE)', fontsize=14, labelpad=10)
    
    for i, p in enumerate(ax.patches):
        ax.annotate(f"{scores[i]:.2f}", 
                   (p.get_x() + p.get_width() / 2., p.get_height()), 
                   ha='center', va='center', 
                   fontsize=12, fontweight='bold', color='black', 
                   xytext=(0, 9), textcoords='offset points')
                   
    sns.despine()
    plt.tight_layout()
    plt.show()